In [0]:
%sql
SELECT
    COUNT(*)                                   AS filas,
    COUNT(DISTINCT id)                         AS modelos_unicos,
    COUNT(DISTINCT ingestion_date)             AS dietas,
    MIN(ingestion_date)                        AS primer_dia,
    MAX(ingestion_date)                        AS ultimo_dia
FROM pf.bronze.models_raw;


In [0]:
%sql

SELECT ingestion_date,
    COUNT(*) AS filas,
    COUNT(DISTINCT id) AS modelos
FROM pf.bronze.models_raw
GROUP BY ingestion_date
ORDER BY ingestion_date;

In [0]:
%sql

SELECT COUNT(*) AS total,
    SUM(
        CASE
            WHEN _id IS NULL THEN 1
            ELSE 0
        END
    ) AS nulos_id_interno,
    SUM(
        CASE
            WHEN id IS NULL THEN 1
            ELSE 0
        END
    ) AS nulos_id,
    SUM(
        CASE
            WHEN downloads IS NULL THEN 1
            ELSE 0
        END
    ) AS nulos_downloads,
    SUM(
        CASE
            WHEN likes IS NULL THEN 1
            ELSE 0
        END
    ) AS nulos_likes,
    SUM(
        CASE
            WHEN createdAt IS NULL THEN 1
            ELSE 0
        END
    ) AS nulos_created,
    SUM(
        CASE
            WHEN lastModified IS NULL THEN 1
            ELSE 0
        END
    ) AS nulos_modified,
    SUM(
        CASE
            WHEN tags IS NULL THEN 1
            ELSE 0
        END
    ) AS nulos_tags,
    SUM(
        CASE
            WHEN payload_json IS NULL THEN 1
            ELSE 0
        END
    ) AS nulos_payload
FROM pf.bronze.models_raw;

In [0]:
%sql
--SIN PARSEO NO ARROJA DATOS COHERENTES
-- SELECT MIN(downloads) AS min_descargas,
--     PERCENTILE(downloads, 0.25) AS p25_descargas,
--     PERCENTILE(downloads, 0.5) AS mediana_descargas,
--     PERCENTILE(downloads, 0.95) AS p95_descargas,
--     MAX(downloads) AS max_descargas,
--     SUM(downloads) AS descargas_totales,
--     PERCENTILE(likes, 0.5) AS mediana_likes,
--     MAX(likes) AS max_likes
-- FROM pf.bronze.models_raw
-- WHERE downloads IS NOT NULL;

In [0]:
%sql
-- VERSION CON CASTEO A DECIMAL PARA OBTENER COHERENTES
WITH base_datos AS (
    SELECT 
        TRY_CAST(downloads AS DECIMAL(18,2)) AS downloads_num,
        TRY_CAST(likes AS DECIMAL(18,2)) AS likes_num
    FROM pf.bronze.models_raw
)
SELECT 
    MIN(downloads_num) AS min_descargas,
    PERCENTILE(downloads_num, 0.25) AS p25_descargas,
    PERCENTILE(downloads_num, 0.5) AS mediana_descargas,
    PERCENTILE(downloads_num, 0.95) AS p95_descargas,
    MAX(downloads_num) AS max_descargas,
    SUM(downloads_num) AS descargas_totales,
    PERCENTILE(likes_num, 0.5) AS mediana_likes,
    MAX(likes_num) AS max_likes
FROM base_datos
WHERE downloads_num IS NOT NULL;

In [0]:
%sql

WITH ultimo AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY id
            ORDER BY ingestion_date DESC
        ) AS rn
    FROM pf.bronze.models_raw
)
SELECT id,
    likes,
    TRY_CAST(downloads AS DECIMAL(18,2)) AS download,
    pipeline_tag,
    library_name,
    createdAt,
    lastModified
FROM ultimo
WHERE rn = 1
ORDER BY download DESC
LIMIT 10;

In [0]:
%sql

SELECT COALESCE(pipeline_tag, '(sin tarea)') AS pipeline_tag,
    COUNT(DISTINCT id) AS modelos,
    SUM(TRY_CAST(downloads AS DECIMAL(18,2))) AS descargas
FROM pf.bronze.models_raw
WHERE ingestion_date = (
        SELECT MAX(ingestion_date)
        FROM pf.bronze.models_raw
    )
GROUP BY pipeline_tag
ORDER BY descargas DESC
LIMIT 15;

In [0]:
%sql

SELECT COALESCE(library_name, '(sin libreria)') AS library_name,
    COUNT(DISTINCT id) AS modelos,
    SUM(TRY_CAST(downloads AS DECIMAL(18,2))) AS descargas
FROM pf.bronze.models_raw
WHERE ingestion_date = (
        SELECT MAX(ingestion_date)
        FROM pf.bronze.models_raw
    )
GROUP BY library_name
ORDER BY descargas DESC
LIMIT 15;

In [0]:
%sql

SELECT private,
    COUNT(*) AS filas
FROM pf.bronze.models_raw
GROUP BY private;
SELECT key,
    cnt
FROM (
        SELECT id || '|' || CAST(ingestion_date AS STRING) AS key,
            COUNT(*) AS cnt
        FROM pf.bronze.models_raw
        GROUP BY id,
            ingestion_date
    )
WHERE cnt > 1
LIMIT 10;

In [0]:
%sql

WITH tags_exploded AS (
    SELECT explode(from_json(tags, 'array<string>')) AS tag
    FROM pf.bronze.models_raw
    WHERE tags IS NOT NULL
)
SELECT tag,
    COUNT(*) AS n
FROM tags_exploded
GROUP BY tag
ORDER BY n DESC
LIMIT 20;

In [0]:
%sql

SELECT COUNT(*) AS con_rescued
FROM pf.bronze.models_raw
WHERE _rescued_data IS NOT NULL;